In [ ]:
import json

In [ ]:
import pickle

In [ ]:
import tqdm

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

False

In [ ]:
import pandas as pd

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

In [ ]:
df = pd.read_excel("data_clean.xlsx", header=0)

In [ ]:
df.head()

,title_rus,title_eng,annotation,description
0,Исследование приоритетов и механизмов реализац...,Study of Priorities and Mechanisms for Impleme...,Работа международных фондов (доноров) должна п...,"Согласно определению международных фондов, про..."
1,Антрополе - научно-популярный видео-подкаст о ...,Anthropole is a Popular Science Video Podcast ...,"\tАнтрополе - научно-популярный проект, в рамк...","Социальное знание близко и интересно обществу,..."
2,"Разработка, создание и ведение сайта, посвящен...","Design, Development and Implementation of a We...",Художественное образование и творчество художн...,Тема обучения арабских художников в художестве...
3,Перевод с английского языка коллективной моног...,Translation from English of the collective mon...,"Коллективная монография, авторы которой являют...","Коллективная монография, авторы которой являют..."
4,Сеть военно-политических союзов в Евразии: баз...,Network of Military in Eurasia: a Database,Проект посвящен изучению сети военно-политичес...,Проект посвящен анализу истории существования ...


In [ ]:
df.tail()

,title_rus,title_eng,annotation,description
1159,Влияние мер контроля за движением капитала на ...,The Impact of Capital Controls on the Investme...,NaN,NaN
1160,"Создание страницы (лендинга) проектов ПУЛ ""Раз...",Creation of a Landing Page for Projects of the...,Проект направлен на разработку и запуск одност...,Проектно-учебная лаборатория «Развитие универс...
1161,Разработка стратегической и контентно-коммуник...,Development of a Strategic and Content-based C...,Проект предполагает разработку системной комму...,Экосистема «Вулканариум–Ойкумена» объединяет н...
1162,Актив Центра развития карьеры - профориентацио...,Active Club of the Career Development Center -...,Актив Центра развития карьеры — это проектная ...,Актуальность проекта «Актив Центра развития ка...
1163,Обзор Методологии в Области Регионоведения. Эт...,Area Studies Methodology Overview. Data Collec...,Данный этап заключается в поиске и сортировке ...,На текущий момент существует разрозненные пред...


In [ ]:
sum(df["title_rus"].isna())

0

In [ ]:
df = df.fillna("")

In [ ]:
df[df.duplicated(subset=["title_rus"])]

,title_rus,title_eng,annotation,description


In [ ]:
df.shape

(1164, 4)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(1164, 4)

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can extract short and concise tags for student projects title in russian.
Please, for a given student project title return a list of corresponding tags, which depicts the field of science to which the project is relevant.
For example, for a project title "Рекомендательная система для студенческих проектов" you answer might be the following list of strings with underscores.
["machine_learning", "recommender_systems", "software_engineering"]
Also another example, for a project title "Прогнозирование эффектов процентной политики Банка России" you answer might be the following list of strings with underscores.
["economics", "macroeconomics", "monetary_policy", "forecasting", "time_series"]
Return strictly a list of strings as an answer.
"""

In [ ]:
messages = []

In [ ]:
for row in df.iterrows():
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row[1]["title_rus"]},
        )
    )

In [ ]:
tags_set = set()

In [ ]:
titles_with_tags_dict = dict()

In [ ]:
for message in tqdm.tqdm(messages):
    text = processor.apply_chat_template(
        message, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(**inputs, max_new_tokens=1024)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    answer = json.loads(processor.parse_response(response)["content"])

    titles_with_tags_dict[message[1]["content"]] = answer
    tags_set = tags_set.union(answer)

100%|██████████| 1164/1164 [36:42<00:00,  1.89s/it]


In [ ]:
with open("titles_with_tags_dict.pkl", "wb") as f:
    pickle.dump(titles_with_tags_dict, f)

In [ ]:
with open("tags_set.pkl", "wb") as f:
    pickle.dump(tags_set, f)

In [ ]:
len(tags_set)

1861

In [ ]:
list(titles_with_tags_dict.values())[0]

['international_relations',
 'political_economics',
 'policy_analysis',
 'BRICS',
 'geopolitics',
 'international_policy',
 'comparative_politics']

In [ ]:
list(titles_with_tags_dict.values())[-1]

['regional_studies', 'methodology', 'data_collection', 'geography']